# COMETS + JupyterLab はじめの一歩（初心者向け）

このノートは、Dockerの `dukovski/comets-lab:1.0` コンテナで動いているJupyterLab上で、COMETSをPython（cometspy）から最小の例で動かすチュートリアルです。

やること：
- 1x1 の「試験管（test tube）」環境を作る
- E. coli の教科書モデル（cobrapyの `textbook`）を読み込む
- COMETSを実行して、バイオマスと培地濃度の推移を可視化する


## 0. 環境チェック

コンテナ内に `cometspy` と `cobra` が入っていること、COMETS本体（Jar）が見えることを確認します。

In [1]:
import os
import sys
from pathlib import Path

candidate_comets_homes = [
    os.environ.get("COMETS_HOME"),
    "/opt/comets_linux",
    "/home/nishioka/comets_linux",
]

comets_home = None
for p in candidate_comets_homes:
    if not p:
        continue
    p = str(p)
    if (Path(p) / "bin").exists():
        comets_home = p
        break

if comets_home is None:
    comets_home = "/opt/comets_linux"

os.environ["COMETS_HOME"] = comets_home

bin_dir = Path(comets_home) / "bin"
jar_candidates = sorted(Path(comets_home).glob("comets_*.jar"))

print("Python:", sys.version)
print("sys.executable:", sys.executable)
print("COMETS_HOME:", comets_home)
print("bin exists:", bin_dir.exists())
print("jar found:", [p.name for p in jar_candidates[:3]])

if not bin_dir.exists():
    raise FileNotFoundError(
        "COMETS_HOME/bin が見つかりません。\n"
        "このノートは COMETS Docker (dukovski/comets-lab:1.0) の中で実行してください。\n"
        "サーバ側で /home/nishioka/IKM_Hiwi/comets.sh で JupyterLab を起動し、そのJupyterでこのノートを開いて実行します。\n"
        "（ローカルPCや別環境のPythonで実行すると /opt/comets_linux が存在せず失敗します）\n"
        f"現在の sys.executable: {sys.executable}\n"
        f"現在の COMETS_HOME: {comets_home}\n"
    )

import cometspy as c
import cobra

print("cobra:", cobra.__version__)
print(
    "cometspy objects:",
    {
        "layout": hasattr(c, "layout"),
        "model": hasattr(c, "model"),
        "params": hasattr(c, "params"),
        "comets": hasattr(c, "comets"),
    },
)


Python: 3.10.19 (main, Feb  4 2026, 20:28:39) [GCC 14.2.0]
sys.executable: /usr/local/bin/python3.10
COMETS_HOME: /opt/comets_linux
bin exists: True
jar found: ['comets_2.12.3.jar']
cobra: 0.30.0
cometspy objects: {'layout': True, 'model': True, 'params': True, 'comets': True}


## 1. レイアウト（試験管）を作る

ここでは 1x1 の世界を作り、培地に糖（グルコース）を入れます。
他の必須成分（アンモニア、リン酸、H2O、H+）は「十分に多い」として大きい値にします。

In [2]:
test_tube = c.layout()
test_tube.grid = [1, 1]

test_tube.set_specific_metabolite("glc__D_e", 0.11)
test_tube.set_specific_metabolite("o2_e", 1000.0)
test_tube.set_specific_metabolite("nh4_e", 1000.0)
test_tube.set_specific_metabolite("pi_e", 1000.0)
test_tube.set_specific_metabolite("h2o_e", 1000.0)
test_tube.set_specific_metabolite("h_e", 1000.0)

test_tube.display_current_media()


building empty layout model
models will need to be added with layout.add_model()
  metabolite  g_refresh  g_static  g_static_val  init_amount    diff_c
0   glc__D_e          0         0             0         0.11  0.000005
1       o2_e          0         0             0      1000.00  0.000005
2      nh4_e          0         0             0      1000.00  0.000005
3       pi_e          0         0             0      1000.00  0.000005
4      h2o_e          0         0             0      1000.00  0.000005
5        h_e          0         0             0      1000.00  0.000005


## 2. FBAモデルを読み込む（E. coli textbook）

cobrapy の `textbook` モデルを読み込み、COMETSで使える形に変換して、レイアウトに追加します。

In [3]:
e_coli_cobra = cobra.io.load_model("textbook")
e_coli = c.model(e_coli_cobra)

e_coli.change_bounds("EX_glc__D_e", -1000, 1000)
e_coli.change_bounds("EX_ac_e", -1000, 1000)
e_coli.change_bounds("ATPM", 8, 1000)

e_coli.initial_pop = [0, 0, 5e-6]
e_coli.obj_style = "MAX_OBJECTIVE_MIN_TOTAL"
e_coli.change_optimizer("GLOP")

test_tube.add_model(e_coli)
test_tube.get_model_ids()


['e_coli_core']

## 3. シミュレーション設定（パラメータ）

まずは短く回る設定にします（`maxCycles` を小さめ）。

In [4]:
sim_params = c.params()
sim_params.set_param("numRunThreads", 1)
sim_params.set_param("defaultVmax", 18)
sim_params.set_param("defaultKm", 0.000003)

sim_params.set_param("maxCycles", 250)
sim_params.set_param("timeStep", 0.02)
sim_params.set_param("spaceWidth", 1)

sim_params.set_param("maxSpaceBiomass", 10)
sim_params.set_param("minSpaceBiomass", 1e-11)

sim_params.set_param("writeMediaLog", True)
sim_params.set_param("deathRate", 0.0)


## 4. 実行

出力先ディレクトリをノートブックの場所（`/workspace`）の下に作ります。

In [5]:
import time

run_dir = Path("comets_runs") / time.strftime("beginner_%Y%m%d_%H%M%S")
run_dir.mkdir(parents=True, exist_ok=True)

experiment = c.comets(test_tube, sim_params, relative_dir=str(run_dir) + "/")
experiment.run()

print("Saved to:", run_dir.resolve())


could not find environmental variable GUROBI_COMETS_HOME or GUROBI_HOME or COMETS_GUROBI_HOME
COMETS will not work with GUROBI until this is solved. 
Here is a solution:
    1. import os and set os.environ['GUROBI_HOME'] then try to make a comets object again
       e.g.   import os
              os.environ['GUROBI_HOME'] = 'C:\\gurobi902\\win64'
These are the expected locations for dependencies:
Dependency 			 expected path
__________ 			 _____________
gurobi			/lib/gurobi.jar

  You have two options to fix this problem:
1.  set each class path correctly by doing:
    comets.set_classpath(libraryname, path)
    e.g.   comets.set_classpath('hamcrest', '/home/chaco001/comets/junit/hamcrest-core-1.3.jar')

    note that versions dont always have to exactly match, but you're on your own if they don't

2.  fully define the classpath yourself by overwriting comets.JAVA_CLASSPATH
       look at the current comets.JAVA_CLASSPATH to see how this should look.

Running COMETS simulation ...
Erro

RuntimeError: COMETS simulation did not complete:
 undetected reason. examine comets.run_output for JAVA trace

## 5. 結果の確認（バイオマス）

In [ ]:
import matplotlib.pyplot as plt

ax = experiment.total_biomass.plot(x="cycle")
ax.set_ylabel("Biomass (gr.)")
plt.show()


## 6. 結果の確認（培地の推移）

培地（media）のログから、代表的な物質の濃度変化を描きます。

In [ ]:
media = experiment.media.copy()
media = media[media.conc_mmol < 900]

targets = {"glc__D_e", "ac_e", "co2_e", "for_e", "o2_e"}
media = media[media.metabolite.isin(targets)]

fig, ax = plt.subplots()
for met, df in media.groupby("metabolite"):
    df.plot(x="cycle", y="conc_mmol", ax=ax, label=met)

ax.set_ylabel("Concentration (mmol)")
plt.show()


## 7. 次にやること（例）

- 嫌気条件にしたい場合：`o2_e` を 0 にして、同じ流れで実行
- 初期バイオマス `e_coli.initial_pop` を変えて挙動を見る
- `maxCycles` や `timeStep` を大きくして長時間シミュレーション


In [ ]:
print("Notebook path:", Path.cwd())
print("Outputs under:", Path("comets_runs").resolve())
